In [15]:
import numpy as np
import pandas as pd
import pandapower as pp
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pickle
import pp_heig_simulation as pp_sim
import pp_heig_plot as pp_plot
from datetime import time
import re

In [16]:
## Import pickle
net_trey = pp.from_pickle("input-data/trey_net_student.p")

In [17]:
### Line parameters
## Add a new column "length_km" to net_trey
net_trey.line["length_km"] = [0.180, 0.100, 0.115, 0.080, 0.090, 0.135, 0.100, 0.205, 0.085, 0.255, 0.340]

## Create new column "cable_type" by removing the unwanted value (_x) used to differentiate each line from "name" with regex
net_trey.line["cable_type"] = net_trey.line.apply(
    lambda x: re.sub(r"_[0-9]$", "", x["name"]), axis=1
)

## Create parameter table for cable types (datasheets)
cable_types = pd.DataFrame(
    {
        "cable_type": ["GKN3x150_150", "GKN3X95_95", "GKN3X50_50", "GKT3X50_50", "GKN3X240_240"],
        "r_ohm_per_km": [0.124, 0.193, 0.387, 0.387, 0.0754],
        "x_ohm_per_km": [0.07, 0.07, 0.07, 0.07, 0.07],
        "c_nf_per_km": [349, 338, 298, 298, 346],
        "max_i_ka": [0.400, 0.252, 0.170, 0.170, 0.512],
    }
)

## Merge WITHOUT destroying pandapower mandatory columns
tmp = net_trey.line[["cable_type"]].merge(cable_types, on="cable_type", how="left")
for col in ["r_ohm_per_km", "x_ohm_per_km", "c_nf_per_km", "max_i_ka"]:
    net_trey.line[col] = tmp[col].values

## Mandatory columns for pandapower
net_trey.line["from_bus"] = net_trey.line["from_bus"].astype(int)
net_trey.line["to_bus"] = net_trey.line["to_bus"].astype(int)
net_trey.line["parallel"] = 1

net_trey.line

,name,from_bus,to_bus,g_us_per_km,df,std_type,in_service,length_km,cable_type,r_ohm_per_km,x_ohm_per_km,c_nf_per_km,max_i_ka,parallel
0,GKN3x150_150,0,1,0.0,1.0,None,True,0.180,GKN3x150_150,0.1240,0.07,349,0.400,1
1,GKN3x150_150_2,1,2,0.0,1.0,None,True,0.100,GKN3x150_150,0.1240,0.07,349,0.400,1
2,GKN3X95_95,1,3,0.0,1.0,None,True,0.115,GKN3X95_95,0.1930,0.07,338,0.252,1
3,GKT3X50_50,3,4,0.0,1.0,None,True,0.080,GKT3X50_50,0.3870,0.07,298,0.170,1
4,GKT3X50_50_2,4,5,0.0,1.0,None,True,0.090,GKT3X50_50,0.3870,0.07,298,0.170,1
5,GKN3x150_150_3,0,6,0.0,1.0,None,True,0.135,GKN3x150_150,0.1240,0.07,349,0.400,1
6,GKN3X95_95_2,6,7,0.0,1.0,None,True,0.100,GKN3X95_95,0.1930,0.07,338,0.252,1
7,GKN3x150_150_4,7,8,0.0,1.0,None,True,0.205,GKN3x150_150,0.1240,0.07,349,0.400,1
8,GKN3X50_50,7,9,0.0,1.0,None,True,0.085,GKN3X50_50,0.3870,0.07,298,0.170,1
9,GKN3X50_50_2,6,10,0.0,1.0,None,True,0.255,GKN3X50_50,0.3870,0.07,298,0.170,1


In [18]:
### Bus nominal voltages
net_trey.bus["vn_kv"] = net_trey.bus.apply(
    lambda row: 18.3 if row["type"] == "Slack" else (0.420 if row["type"] == "PQ" else row.get("vn_kv", np.nan)),
    axis=1,
)

# Creat ext_grid 
net_trey.ext_grid.drop(net_trey.ext_grid.index, inplace=True)

slack_bus = int(net_trey.bus.index[net_trey.bus["type"] == "Slack"][0])
pp.create_ext_grid(net_trey, bus=slack_bus, vm_pu=1.0)
net_trey.ext_grid["bus"] = net_trey.ext_grid["bus"].astype(int)


net_trey.bus

,name,type,zone,in_service,vn_kv
0,STMT003438,PQ,Trafo,True,0.42
1,CDBT004764,PQ,North,True,0.42
2,CDBT003746,PQ,North,True,0.42
3,CDBT004760,PQ,North,True,0.42
4,CDBT012139,PQ,North,True,0.42
5,CDBT900784,PQ,North,True,0.42
6,CDBT901452,PQ,South,True,0.42
7,CDBT004774,PQ,South,True,0.42
8,CDBT901604,PQ,South,True,0.42
9,CDBT016055,PQ,South,True,0.42


In [19]:
### Loads mandatory columns for pandapower / time series
net_trey.load["bus"] = net_trey.load["bus"].astype(int)
net_trey.load["p_mw"] = 0.01
net_trey.load["q_mvar"] = 0.005
net_trey.load["scaling"] = 1.0
net_trey.load["profile_mapping"] = 1.0

net_trey.load


,name,bus,const_z_percent,const_i_percent,sn_mva,in_service,type,p_mw,q_mvar,scaling,profile_mapping
0,STMT003438,0,0.0,0.0,None,True,wye,0.01,0.005,1.0,1.0
1,CDBT004764,1,0.0,0.0,None,True,wye,0.01,0.005,1.0,1.0
2,CDBT003746,2,0.0,0.0,None,True,wye,0.01,0.005,1.0,1.0
3,CDBT004760,3,0.0,0.0,None,True,wye,0.01,0.005,1.0,1.0
4,CDBT012139,4,0.0,0.0,None,True,wye,0.01,0.005,1.0,1.0
5,CDBT900784,5,0.0,0.0,None,True,wye,0.01,0.005,1.0,1.0
6,CDBT901452,6,0.0,0.0,None,True,wye,0.01,0.005,1.0,1.0
7,CDBT004774,7,0.0,0.0,None,True,wye,0.01,0.005,1.0,1.0
8,CDBT901604,8,0.0,0.0,None,True,wye,0.01,0.005,1.0,1.0
9,CDBT016055,9,0.0,0.0,None,True,wye,0.01,0.005,1.0,1.0


In [20]:
## Plot grid
pp_plot.plot_power_network(
    net=net_trey,
    plot_title="Trey",
    filename="trey_grid_example",
)

In [ ]:
### Time series
profile_file_path = "input-data/load_curve/power_profile_cabinet_summer_week.xlsx"
time_series = pp_sim.load_power_profile_form_xlsx(file_path=profile_file_path)

# Apply time series to net_trey
net_trey.load["profile_mapping"] = net_trey.load["bus"].astype(int)

pp_sim.apply_power_profile(net=net_trey, equipment="load", power_profiles=time_series["load"])

pp_sim.create_output_writer(net=net_trey, add_results=["res_line.p_from_mw"])

result_df = pp_sim.run_time_simulation(net=net_trey)

pp_plot.plot_timeseries_result(
    data_df=result_df["res_bus.vm_pu"],
    ylabel="V [pu]",
    plot_title="Tension des bus",
    filename="voltage_result_trey",
)

pp_plot.plot_timeseries_result(
    data_df=result_df["res_line.p_from_mw"],
    ylabel="P [MW]",
    plot_title="Puissance des lignes",
    filename="line_power_result_trey",
)

/home/max/github/rhtlab/.venv/lib/python3.12/site-packages/pandapower/timeseries/output_writer.py:177: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'range(0, 96)' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.



TypeError: 'module' object is not callable